# Fine-tuning Médical — TechCorp IA Hackathon

Fine-tuning QLoRA de `microsoft/Phi-3.5-mini-instruct` sur le dataset médical.

**Source de données (par priorité) :**
1. `medical_dataset/medical_dataset_prepared.json` — uploadé sur Colab depuis le repo
2. Téléchargement automatique depuis `ruslanmv/ai-medical-chatbot` (HuggingFace) si fichier absent

**Infrastructure :** Google Colab Pro (GPU T4/A100) — modèle expérimental, pas pour production.

---
## Étape 1 — Installation des dépendances

In [ ]:
!pip install -q transformers==4.44.0 peft==0.12.0 bitsandbytes==0.43.3 \
    trl==0.11.1 datasets==3.0.1 accelerate==0.34.2
print("Dependances installees")

## Étape 2 — Imports et configuration

Détection automatique du GPU disponible et configuration du dtype optimal.

In [ ]:
import torch
import json
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset, Dataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
BASE_MODEL = "microsoft/Phi-3.5-mini-instruct"
OUTPUT_DIR = "/content/medical_model"
LOCAL_DATASET = "/content/medical_dataset_prepared.json"

print(f"Device  : {DEVICE}")
print(f"dtype   : {DTYPE}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Étape 3 — Chargement du dataset médical

**Option A (recommandée) :** Uploader `medical_dataset/medical_dataset_prepared.json` dans Colab  
→ Panneau Fichiers (icône dossier) → glisser le fichier → le renommer `/content/medical_dataset_prepared.json`

**Option B (fallback automatique) :** Téléchargement depuis HuggingFace si le fichier est absent.

Le dataset préparé est généré par `rendu/data/prepare_medical.py` et stocké dans `medical_dataset/`.

In [ ]:
DATASET_LIMIT = 5000

if os.path.exists(LOCAL_DATASET):
    # Option A : dataset local prepare par l'equipe DATA (medical_dataset/)
    print(f"[LOCAL] Chargement depuis {LOCAL_DATASET}...")
    with open(LOCAL_DATASET, encoding="utf-8") as f:
        raw_data = json.load(f)
    raw_data = raw_data[:DATASET_LIMIT]
    raw_dataset = Dataset.from_list([
        {"instruction": r["instruction"], "output": r["output"]}
        for r in raw_data
        if r.get("instruction") and r.get("output")
    ])
    print(f"[LOCAL] {len(raw_dataset)} entrees chargees depuis medical_dataset/")
else:
    # Option B : telechargement HuggingFace (fallback)
    print(f"[HUGGINGFACE] Telechargement ruslanmv/ai-medical-chatbot (limite: {DATASET_LIMIT})...")
    hf = load_dataset("ruslanmv/ai-medical-chatbot", split="train")
    hf = hf.select(range(min(DATASET_LIMIT, len(hf))))
    raw_dataset = hf.map(
        lambda x: {
            "instruction": (x.get("Patient") or "").strip(),
            "output": (x.get("Doctor") or "").strip()
        },
        remove_columns=hf.column_names
    )
    print(f"[HUGGINGFACE] {len(raw_dataset)} entrees chargees")

print(f"\nExemple :")
print(f"  instruction : {raw_dataset[0]['instruction'][:120]}")
print(f"  output      : {raw_dataset[0]['output'][:120]}")

## Étape 4 — Formatage au format ChatML (Phi-3.5)

Phi-3.5 utilise le template : `<|user|>\n{question}<|end|>\n<|assistant|>\n{réponse}<|end|>`  
Ce format s'applique identiquement que le dataset vienne du fichier local ou de HuggingFace.

In [ ]:
def format_chatml(example):
    q = (example.get("instruction") or "").strip()
    a = (example.get("output") or "").strip()
    if not q or not a:
        return {"text": None}
    return {"text": f"<|user|>\n{q}<|end|>\n<|assistant|>\n{a}<|end|>"}

formatted = raw_dataset.map(format_chatml, remove_columns=raw_dataset.column_names)
formatted = formatted.filter(lambda x: x["text"] is not None)

print(f"{len(formatted)} exemples formatés")
print(f"\nExemple formatte :")
print(formatted[0]["text"][:300])

## Étape 5 — Chargement du modèle de base avec QLoRA 4-bit

QLoRA (Quantized LoRA) permet d'entraîner Phi-3.5-mini (3.8B) en 4-bit sur ~8GB de VRAM.  
La quantification NF4 préserve les performances tout en réduisant la mémoire de ~75%.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Chargement {BASE_MODEL} en 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)
print(f"Modele charge — Parametres totaux: {model.num_parameters():,}")

## Étape 6 — Configuration LoRA

LoRA (Low-Rank Adaptation) ajoute des matrices d'adaptation de rang faible (r=8) sur les couches d'attention.  
Seuls ~0.5% des paramètres sont entraînés, ce qui accélère l'entraînement de ~10x.

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["qkv_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
trainable, total = model.get_nb_trainable_parameters()
print(f"LoRA applique")
print(f"  Parametres entrainables : {trainable:,} ({100*trainable/total:.2f}%)")
print(f"  Parametres totaux       : {total:,}")

## Étape 7 — Configuration de l'entraînement (SFTTrainer)

SFT (Supervised Fine-Tuning) avec `trl.SFTTrainer`.  
- 2 epochs pour rester dans les limites Colab
- Gradient accumulation x4 → batch effectif de 8
- `max_seq_length=512` adapté aux conversations médicales

In [ ]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=25,
    save_steps=200,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    max_seq_length=512,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=formatted,
    processing_class=tokenizer,
)

steps_per_epoch = len(formatted) // (sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps)
print(f"Trainer configure")
print(f"  Exemples         : {len(formatted)}")
print(f"  Steps par epoch  : {steps_per_epoch}")
print(f"  Steps totaux     : {steps_per_epoch * sft_config.num_train_epochs}")

## Étape 8 — Entraînement

La loss doit descendre progressivement. Une loss finale < 1.5 indique un bon apprentissage.  
Durée estimée : ~30-45 min (T4), ~15-20 min (A100).

In [ ]:
import time

print("Demarrage de l'entrainement...\n")
t0 = time.time()

train_result = trainer.train()

elapsed = time.time() - t0
print(f"\nEntrainement termine en {elapsed/60:.1f} minutes")
print(f"  Loss finale      : {train_result.training_loss:.4f}")
print(f"  Steps effectues  : {train_result.global_step}")
for log in trainer.state.log_history:
    if "loss" in log:
        print(f"    step={log.get('step', '?'):4d}  epoch={log.get('epoch', '?'):.2f}  loss={log['loss']:.4f}")

## Étape 9 — Sauvegarde du modèle

Sauvegarde de l'adapter LoRA et du tokenizer dans `/content/medical_model/`.  
Ce répertoire peut être téléchargé ou monté sur Google Drive pour conservation.

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

files = os.listdir(OUTPUT_DIR)
print(f"Modele sauvegarde dans {OUTPUT_DIR}")
print(f"  Fichiers : {files}")

# Optionnel : monter sur Google Drive pour conserver apres fermeture Colab
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/medical_model /content/drive/MyDrive/techcorp_medical_model

## Étape 10 — Test du modèle fine-tuné

Validation qualitative avec 3 questions médicales représentatives.  
Les réponses doivent être précises, contextualisées et sans hallucinations grossières.

In [ ]:
print("Chargement du modele fine-tune pour test...")
test_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
test_model = PeftModel.from_pretrained(test_model, OUTPUT_DIR)
test_model.eval()

MEDICAL_QUESTIONS = [
    "I have a persistent headache for 3 days with sensitivity to light. What could it be?",
    "What are the main differences between type 1 and type 2 diabetes?",
    "My child has a fever of 38.5C and a sore throat. Should I be worried?",
]

def generate(question, max_new_tokens=200):
    prompt = f"<|user|>\n{question}<|end|>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = test_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

print("\n" + "="*60)
for i, q in enumerate(MEDICAL_QUESTIONS, 1):
    print(f"\n[{i}] Patient: {q}")
    print(f"    Doctor : {generate(q)[:400]}")
    print("-"*60)

logs = [l for l in trainer.state.log_history if "loss" in l]
print(f"\nResume : loss initiale={logs[0]['loss']:.4f}  finale={logs[-1]['loss']:.4f}  duree={elapsed/60:.1f}min")
print("Modele experimental valide (non destine a la production)")